In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.model_selection import StratifiedKFold
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [ ]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
n_components = 32
n_splits = 5

In [ ]:
rng = np.random.RandomState(seed)

In [ ]:
data = pd.read_parquet("../data/GBG500.parquet")
data

In [ ]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

In [ ]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [ ]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

In [ ]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

In [ ]:
# 5-fold stratified CV with PCA
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
scores = {k: np.full(len(y_true), np.nan) for k in ['lof', 'iso_forest', 'ocsvm']}

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    pca = PCA(n_components=n_components, random_state=rng.randint(1000))
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)
    print(f"  Variance explained (train): {sum(pca.explained_variance_ratio_):.4f}")

    lof = LocalOutlierFactor(n_neighbors=20, novelty=True)
    lof.fit(X_train)
    scores['lof'][test_idx] = -lof.score_samples(X_test)

    iso = IsolationForest(random_state=rng.randint(1000))
    iso.fit(X_train)
    scores['iso_forest'][test_idx] = -iso.score_samples(X_test)

    ocsvm = OneClassSVM(kernel='rbf')
    ocsvm.fit(X_train)
    scores['ocsvm'][test_idx] = -ocsvm.decision_function(X_test)

    print(f"Fold {fold+1}/{n_splits} done")

In [ ]:
for key, glue_name in [('lof', 'GBG500_ap_spectral_pca_lof'),
                        ('iso_forest', 'GBG500_ap_spectral_pca_iso_forest'),
                        ('ocsvm', 'GBG500_ap_spectral_pca_ocsvm')]:
    ap = average_precision_score(y_true, scores[key])
    print(f"PCA+{key} AP = {ap:.4f}")
    sb.glue(glue_name, float(ap))